In [2]:
import numpy as np
import pandas as pd

In [3]:
gas_data = pd.read_csv('Data/DB elder monitoring/database_gas.csv')
pos_data = pd.read_csv('Data/DB elder monitoring/database_pos.csv')

In [4]:
# --- Helper: min-max normalize ---
def normalize(series):
    return (series - series.min()) / (series.max() - series.min())

def add_synthetic_mq135(df):
    # Normalize each field
    df_norm = pd.DataFrame()
    fields = [
        "CO2MG811Value",
        "CO2CosIRValue",
        "COValue",
        "MOX1",
        "MOX2",
        "MOX3",
        "MOX4"
    ]

    for f in fields:
        df_norm[f] = normalize(df[f])

    # Weighted synthetic MQ-135
    df["MQ135_synth"] = (
        0.25 * df_norm["CO2MG811Value"] +
        0.20 * df_norm["CO2CosIRValue"] +
        0.15 * df_norm["COValue"] +
        0.10 * df_norm["MOX1"] +
        0.10 * df_norm["MOX2"] +
        0.10 * df_norm["MOX3"] +
        0.10 * df_norm["MOX4"]
    )

    # Convert 0–1 to ESP32 12-bit ADC (0–4095)
    df["MQ135_ADC"] = (df["MQ135_synth"] * 4095).astype(int)

    return df

In [7]:
mq135_df = add_synthetic_mq135(gas_data)
gas_data['MQ-135 Value'] = mq135_df['MQ135_ADC']

In [13]:
gas_data['MQ-135 Value'].min()

np.int64(796)